# 🏥 Heal Your Heart EHR — Remote Ollama GPU Host

This notebook runs **Ollama** and the verified **`qwen2.5:7b`** model on a **Free Google Colab T4 GPU**.

It exposes a secure public HTTPS URL via Cloudflare Tunnel so teammates with weak laptops can develop the EHR AI Agent inside Antigravity without overloading their machines.

### Instructions:
1. Go to **Runtime** > **Change runtime type** > Select **T4 GPU** > Click **Save**.
2. Click the **Play (Run)** button on the cell below.
3. Copy the printed `OLLAMA_BASE_URL` into your local `backend/.env` file.
4. Keep this Colab browser tab open while you work.

In [ ]:
# Step 1: Install Ollama
!curl -fsSL https://ollama.com/install.sh | sh

# Step 2: Start Ollama service in background
import subprocess, time, re
ollama_proc = subprocess.Popen(["ollama", "serve"])
time.sleep(3)

# Step 3: Pull verified EHR model (qwen2.5:7b)
!ollama pull qwen2.5:7b

# Step 4: Install Cloudflare Tunnel (cloudflared)
!wget -q -nc https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64.deb
!dpkg -i cloudflared-linux-amd64.deb > /dev/null 2>&1

# Step 5: Start Tunnel and print connection link for .env
print("\n🚀 Starting Cloudflare Tunnel... Generating public connection link...")
tunnel_proc = subprocess.Popen(
    ["cloudflared", "tunnel", "--url", "http://127.0.0.1:11434"],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)

for line in iter(tunnel_proc.stderr.readline, ''):
    match = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if match:
        public_url = match.group(0)
        print("\n" + "="*60)
        print("🎉 OLLAMA GPU HOST IS ONLINE AND READY!")
        print("="*60)
        print("\n👉 Copy & paste this into your local 'backend/.env' file:\n")
        print(f"OLLAMA_BASE_URL={public_url}")
        print("OLLAMA_MODEL=qwen2.5:7b\n")
        print("="*60)
        print("Keep this Colab tab running while working in Antigravity.")
        print("="*60)
        break

# Keep process alive
try:
    tunnel_proc.wait()
except KeyboardInterrupt:
    tunnel_proc.terminate()
    ollama_proc.terminate()
    print("Server stopped.")